# Notebook 03: tiny scalar backward engine

In notebook 02, we computed the chain rule by hand. In this notebook, we build a tiny teaching engine that stores the same forward values and backward sensitivities inside Python objects. The goal is not to replace the production MLP code; it is to make the idea behind `backward()` visible.

## Start with the manual expression

First, repeat the same expression from notebook 02. These plain floats give us a baseline: we know the forward values should be `d = -6.0`, `e = 4.0`, and `f = 16.0`. Later, our `Value` objects should produce the same forward numbers while also remembering the graph.

In [32]:
a = 2.0
b = -3.0
c = 10.0

d = a * b
e = d + c
f = e * e

print(d, e, f)

-6.0 4.0 16.0


## From hand bookkeeping to object bookkeeping

The manual backward pass worked because we kept track of each intermediate value and each local derivative. A `Value` object will keep that bookkeeping next to the number itself. For now, it stores the forward number in `data`, the backward sensitivity in `grad`, the parent values in `_prev`, and the operation name in `_op`.

## First version of `Value`

This first version only supports storage, printing, and addition. Addition creates a new `Value`, stores the computed number, and remembers the two parent `Value` objects that produced it. No gradients move yet; we are only building the graph.

In [33]:
class Value:
    def __init__(
        self,
        data: float,
        _children: tuple["Value", ...] = (),
        _op: str = "",
    ) -> None:
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op

    def __repr__(self) -> str:
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other: "Value") -> "Value":
        return Value(data=self.data + other.data, _children=(self, other), _op="+")

    def __mul__(self, other: "Value") -> "Value":
        return Value(data=self.data * other.data, _children=(self, other), _op="*")

## Leaf values

A leaf value is an input we create directly, like `a = Value(2.0)`. It has a number in `data`, starts with `grad = 0.0`, and has no parents because no earlier operation created it.

In [34]:
a = Value(2.0)
b = Value(-3.0)

print(a)
print(b)

Value(data=2.0, grad=0.0)
Value(data=-3.0, grad=0.0)


## Addition creates a graph node

When we write `c = a + b`, Python calls `a.__add__(b)`. The result `c` stores the forward value `-1.0`, remembers that the operation was `+`, and keeps links back to both parent objects. Those links are what the backward pass will use later to update `a.grad` and `b.grad`.

In [35]:
a = Value(2.0)
b = Value(-3.0)
c = a + b

print(c)
print(c._op)
print(c._prev)

Value(data=-1.0, grad=0.0)
+
{Value(data=-3.0, grad=0.0), Value(data=2.0, grad=0.0)}


## Inspect the parent links

The parent links store the actual `Value` objects, not just the numbers `2.0` and `-3.0`. Printing only each parent's `.data` is a gentle way to check which forward values are connected to `c`. Because `_prev` is a set, the order may change, but both parents should be present.

In [36]:
print(c.data)
print([parent.data for parent in c._prev])

-1.0
[-3.0, 2.0]


In [37]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

d = a * b
e = d + c

print(d)
print(e)
print(e._op)
print([parent.data for parent in e._prev])

Value(data=-6.0, grad=0.0)
Value(data=4.0, grad=0.0)
+
[10.0, -6.0]


In [39]:
f = e * e

print(f)
print(f._op)
print([parent.data for parent in f._prev])

Value(data=16.0, grad=0.0)
*
[4.0]
